# KuchoLM training

CC100-ja から NIDA_FICTION 学習データを生成し、SentencePiece → Transformer 学習 → 保存 → 推論までこの notebook だけで実行します。

`/content/kucholm_nida.jsonl` が無ければ自動生成します。

In [ ]:
!pip -q install mecab-python3 unidic-lite datasets sentencepiece torch

## 1. 設定

In [ ]:
from pathlib import Path
import json, math, random, re
import MeCab
import sentencepiece as spm
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset

DATA_PATH = Path('/content/kucholm_nida.jsonl')
WORK_DIR = Path('/content/kucholm_work')
WORK_DIR.mkdir(parents=True, exist_ok=True)
MAX_ROWS = 100_000
DATASET_NAME = 'range3/cc100-ja'
DATASET_SPLIT = 'train'
TEXT_COLUMN = 'text'
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tagger = MeCab.Tagger()
print('device:', device)

## 2. NIDA_FICTION データ生成

ここで CC100-ja を読み込み、`/content/kucholm_nida.jsonl` を最大10万件生成します。既にファイルがある場合は再生成しません。

In [ ]:
URL_RE = re.compile(r'https?://|www\.|```|`[^`]+`')
SENTENCE_SPLIT_RE = re.compile(r'(.+?[。！？!?]+|.+$)', re.S)

def parse_tokens(text):
    node = tagger.parseToNode(text)
    tokens = []
    while node:
        if node.surface:
            f = node.feature.split(',')
            tokens.append({'surface': node.surface, 'pos': f[0] if len(f)>0 else '', 'ctype': f[4] if len(f)>4 else '*', 'lemma': f[7] if len(f)>7 else '*', 'orth_base': f[10] if len(f)>10 else '*'})
        node = node.next
    return tokens

def dictionary_form(token):
    for key in ('orth_base', 'lemma'):
        value = token.get(key, '*')
        if value not in {'', '*'} and re.search(r'[ぁ-ん一-龯]', value):
            return value
    return token['surface']

def is_ichidan(token, base):
    ctype = token.get('ctype', '')
    return '下一段' in ctype or '上一段' in ctype or '一段' in ctype or (base.endswith('る') and token.get('surface', '') == base[:-1])

def ta_form(base, token):
    if base == '行く': return '行った'
    if base == '来る': return '来た'
    if base == 'する': return 'した'
    if is_ichidan(token, base): return base[:-1] + 'た'
    if base.endswith(('う','つ','る')): return base[:-1] + 'った'
    if base.endswith(('む','ぶ','ぬ')): return base[:-1] + 'んだ'
    if base.endswith('く'): return base[:-1] + 'いた'
    if base.endswith('ぐ'): return base[:-1] + 'いだ'
    if base.endswith('す'): return base[:-1] + 'した'
    return base + 'た'

def nai_form(base, token):
    if base == 'する': return 'しない'
    if base == '来る': return '来ない'
    if is_ichidan(token, base): return base[:-1] + 'ない'
    if base.endswith('う'): return base[:-1] + 'わない'
    mp = {'く':'か','ぐ':'が','す':'さ','つ':'た','ぬ':'な','ぶ':'ば','む':'ま','る':'ら'}
    return base[:-1] + mp[base[-1]] + 'ない' if base[-1:] in mp else base + 'ない'

def soften_surface(text):
    for p, r in [(r'ということです$','ってこと'),(r'ということでした$','ってことだった'),(r'のであります$','んだ'),(r'であります$','なんだ'),(r'なのです$','なんだ'),(r'のです$','んだ'),(r'でしょう$','だろう'),(r'ではありません$','じゃない'),(r'ではないです$','じゃない'),(r'ではない$','じゃない')]:
        text = re.sub(p, r, text)
    return text

def auxiliary_tail(body):
    for p, r in [(r'てきちゃいました$','てきちゃった'),(r'て来ちゃいました$','て来ちゃった'),(r'てきました$','てきた'),(r'て来ました$','て来た'),(r'てこられました$','てこられた'),(r'て来られました$','て来られた'),(r'ていきました$','ていった'),(r'て行きました$','て行った'),(r'てしまいました$','てしまった'),(r'でしまいました$','でしまった'),(r'できました$','できた')]:
        if re.search(p, body): return re.sub(p, r, body)
    return None

def convert_polite_tail(body):
    for p, r in [(r'かもしれません$','かもしれない'),(r'わかりません$','わからない'),(r'知りません$','知らない'),(r'いけません$','いけない'),(r'ありません$','ない'),(r'ございました$','あった'),(r'ございます$','ある')]:
        if re.search(p, body): return re.sub(p, r, body)
    aux = auxiliary_tail(body)
    if aux is not None: return aux
    tokens = parse_tokens(body)
    if not tokens: return body
    surfaces = [t['surface'] for t in tokens]
    particle = ''
    if surfaces and surfaces[-1] in {'ね','よ','な'}:
        particle = surfaces.pop(); tokens = tokens[:-1]
    for suffix, mode in [(['ませ','ん','でし','た'],'negative_past'),(['ませ','ん'],'negative'),(['まし','た'],'past'),(['ます'],'present')]:
        if len(surfaces) < len(suffix) or surfaces[-len(suffix):] != suffix: continue
        suffix_start = len(tokens) - len(suffix)
        vi = next((i for i in range(suffix_start-1, -1, -1) if tokens[i]['pos'] == '動詞'), None)
        if vi is None: continue
        verb = tokens[vi]; base = dictionary_form(verb); prefix = ''.join(t['surface'] for t in tokens[:vi])
        chain = ''.join(t['surface'] for t in tokens[max(0,vi-2):vi+1])
        if any(x in chain for x in ('られ','され','こられ','おられ')):
            stem = ''.join(t['surface'] for t in tokens[:suffix_start])
            if mode == 'past': return stem + 'た' + particle
            if mode == 'present': return stem + particle
        if mode == 'present': repl = base
        elif mode == 'past': repl = ta_form(base, verb)
        else:
            neg = nai_form(base, verb); repl = neg if mode == 'negative' else neg[:-2] + 'なかった'
        return prefix + repl + particle
    if surfaces[-2:] == ['でし','た']: return ''.join(surfaces[:-2]) + 'だった' + particle
    if surfaces[-1:] == ['です']: return ''.join(surfaces[:-1]) + particle
    return body

def convert_sentence(sentence):
    m = re.match(r'^(\s*)(.*?)(\s*)$', sentence, re.S)
    leading, core, trailing = m.groups()
    if not core or URL_RE.search(core): return sentence
    pm = re.search(r'([。！？!?]+)$', core)
    punctuation = pm.group(1) if pm else ''
    body = core[:-len(punctuation)] if punctuation else core
    q = bool(re.search(r'[？?]$', punctuation))
    body = soften_surface(body)
    if body.endswith('か') and q: body = body[:-1]
    converted = convert_polite_tail(body)
    if converted.endswith(('ね','よ','な')):
        p = converted[-1]; converted = converted[:-1] + 'ニダ' + p
    else:
        converted += 'ニカ' if q else ('ニダね' if re.search(r'(ない|難しい|心配|残念|大丈夫)$', converted) else 'ニダよ')
    return leading + converted + punctuation + trailing

def to_nida(text):
    if not text or URL_RE.search(text): return None
    parts = []; cursor = 0
    for m in SENTENCE_SPLIT_RE.finditer(text):
        if m.start() > cursor: parts.append(text[cursor:m.start()])
        parts.append(convert_sentence(m.group(0))); cursor = m.end()
    if cursor < len(text): parts.append(text[cursor:])
    return ''.join(parts)

if not DATA_PATH.exists():
    print('Generating NIDA_FICTION JSONL...')
    dataset = load_dataset(DATASET_NAME, split=DATASET_SPLIT, streaming=True)
    written = 0
    with DATA_PATH.open('w', encoding='utf-8') as out:
        for row in dataset:
            source = str(row[TEXT_COLUMN])
            if not source or len(source.strip()) < 2 or len(source) > 256: continue
            target = to_nida(source)
            if not target or target == source: continue
            out.write(json.dumps({'style':'NIDA_FICTION','source':source,'target':target}, ensure_ascii=False) + '\n')
            written += 1
            if written >= MAX_ROWS: break
    print('written:', written)
    print('saved:', DATA_PATH)
    print('size MB:', DATA_PATH.stat().st_size / 1024 / 1024)
else:
    print('using existing:', DATA_PATH)

## 3. 学習データ読み込み

In [ ]:
rows=[]
with DATA_PATH.open(encoding='utf-8') as f:
    for line in f:
        x=json.loads(line); rows.append((f"<NIDA_FICTION> {x['source']}",x['target']))
random.shuffle(rows)
cut=max(1,int(len(rows)*0.98)); train_rows=rows[:cut]; val_rows=rows[cut:]
print('train:',len(train_rows),'val:',len(val_rows))

## 4. SentencePiece

In [ ]:
spm_input=WORK_DIR/'spm_train.txt'
with spm_input.open('w',encoding='utf-8') as f:
    for src,tgt in train_rows:
        f.write(src.replace('\n',' ')+'\n'); f.write(tgt.replace('\n',' ')+'\n')
spm.SentencePieceTrainer.train(input=str(spm_input),model_prefix=str(WORK_DIR/'kucholm_spm'),vocab_size=8000,model_type='bpe',character_coverage=0.9995,pad_id=0,unk_id=1,bos_id=2,eos_id=3,user_defined_symbols=['<NIDA_FICTION>'])
sp=spm.SentencePieceProcessor(model_file=str(WORK_DIR/'kucholm_spm.model')); PAD,UNK,BOS,EOS=0,1,2,3; VOCAB=sp.vocab_size(); print('vocab:',VOCAB)

## 5. Dataset / DataLoader

In [ ]:
MAX_LEN=128
BATCH=32 if device.type=='cuda' else 8
def encode(text): return [2]+sp.encode(text,out_type=int)[:MAX_LEN-2]+[3]
class PairDataset(Dataset):
    def __init__(self,data): self.data=data
    def __len__(self): return len(self.data)
    def __getitem__(self,i):
        a,b=self.data[i]; return torch.tensor(encode(a)),torch.tensor(encode(b))
def collate(batch):
    a,b=zip(*batch); return nn.utils.rnn.pad_sequence(a,batch_first=True,padding_value=0),nn.utils.rnn.pad_sequence(b,batch_first=True,padding_value=0)
train_loader=DataLoader(PairDataset(train_rows),batch_size=BATCH,shuffle=True,collate_fn=collate)
val_loader=DataLoader(PairDataset(val_rows),batch_size=BATCH,shuffle=False,collate_fn=collate)

## 6. KuchoLM NIDA-15M

In [ ]:
D_MODEL=320
NHEAD=8
ENC_LAYERS=4
DEC_LAYERS=4
FF=1280
DROPOUT=0.1
EPOCHS=5
LR=2.5e-4
class KuchoTransformer(nn.Module):
    def __init__(self):
        super().__init__(); self.embed=nn.Embedding(VOCAB,D_MODEL,padding_idx=0); self.pos=nn.Embedding(MAX_LEN,D_MODEL)
        self.tf=nn.Transformer(d_model=D_MODEL,nhead=NHEAD,num_encoder_layers=ENC_LAYERS,num_decoder_layers=DEC_LAYERS,dim_feedforward=FF,dropout=DROPOUT,batch_first=True,norm_first=True)
        self.lm_head=nn.Linear(D_MODEL,VOCAB,bias=False); self.lm_head.weight=self.embed.weight
    def add_pos(self,x):
        p=torch.arange(x.size(1),device=x.device).unsqueeze(0); return self.embed(x)*math.sqrt(D_MODEL)+self.pos(p)
    def forward(self,src,tgt_in):
        src_pad=src.eq(0); tgt_pad=tgt_in.eq(0); mask=nn.Transformer.generate_square_subsequent_mask(tgt_in.size(1),device=tgt_in.device)
        h=self.tf(self.add_pos(src),self.add_pos(tgt_in),tgt_mask=mask,src_key_padding_mask=src_pad,tgt_key_padding_mask=tgt_pad,memory_key_padding_mask=src_pad)
        return self.lm_head(h)
model=KuchoTransformer().to(device); print(f'{sum(p.numel() for p in model.parameters())/1e6:.2f}M parameters')

## 7. 学習

In [ ]:
optimizer=torch.optim.AdamW(model.parameters(),lr=LR,betas=(0.9,0.98),weight_decay=0.01)
criterion=nn.CrossEntropyLoss(ignore_index=0,label_smoothing=0.05)
scaler=torch.amp.GradScaler('cuda',enabled=device.type=='cuda')
best_val=float('inf'); best_path=WORK_DIR/'KuchoLM-NIDA-15M.pt'
for epoch in range(1,EPOCHS+1):
    model.train(); train_loss=0.0
    for src,tgt in train_loader:
        src,tgt=src.to(device),tgt.to(device); optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda',enabled=device.type=='cuda'):
            logits=model(src,tgt[:,:-1]); loss=criterion(logits.reshape(-1,VOCAB),tgt[:,1:].reshape(-1))
        scaler.scale(loss).backward(); scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(optimizer); scaler.update(); train_loss+=loss.item()
    model.eval(); val_loss=0.0
    with torch.no_grad():
        for src,tgt in val_loader:
            src,tgt=src.to(device),tgt.to(device); logits=model(src,tgt[:,:-1]); val_loss+=criterion(logits.reshape(-1,VOCAB),tgt[:,1:].reshape(-1)).item()
    train_loss/=max(1,len(train_loader)); val_loss/=max(1,len(val_loader)); print(f'epoch {epoch}: train={train_loss:.4f} val={val_loss:.4f}')
    if val_loss<best_val:
        best_val=val_loss; torch.save({'model':model.state_dict(),'best_val':best_val},best_path); print('saved best:',best_path)

## 8. 推論

In [ ]:
checkpoint=torch.load(best_path,map_location=device); model.load_state_dict(checkpoint['model']); model.eval()
@torch.no_grad()
def infer(text,max_new_tokens=96,repetition_penalty=1.15):
    src=torch.tensor([encode('<NIDA_FICTION> '+text)],device=device); out=[2]
    max_steps=min(max_new_tokens,max(12,src.size(1)+24),MAX_LEN-1)
    for _ in range(max_steps):
        logits=model(src,torch.tensor([out],device=device))[0,-1].clone()
        for token_id in set(out[1:]):
            logits[token_id]=logits[token_id]/repetition_penalty if logits[token_id]>0 else logits[token_id]*repetition_penalty
        next_id=int(torch.argmax(logits))
        if next_id==3: break
        out.append(next_id)
    return sp.decode(out[1:])
for s in ['今日は学校です。','明日は雨が降るかもしれません。','最近少し暖かくなってきました。']:
    print(s,'->',infer(s))
print('saved:',best_path)